# Data Cleaning Notebook – Nova Retail Group

**Project:** Business KPI Dashboard & Data Modeling  
**Author:** Daisy Omondi  
**Tools:** Python, Pandas, SQL-ready data modeling, Power BI

## Purpose

This notebook shows how raw retail data can be cleaned and prepared for a business intelligence dashboard.

The goal is not only to clean data, but to build a reliable reporting foundation for management KPIs such as revenue, profit, profit margin, number of orders, average order value and customer count.


## 1. Import Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

print("Libraries loaded successfully.")

## 2. Project Folders

The notebook expects raw data in `data/raw`.

If no raw files exist, it creates small fictional demo data so the project can still run for portfolio reviewers.


In [ ]:
PROJECT_ROOT = Path.cwd()

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DATA_DIR = PROJECT_ROOT / "data" / "cleaned"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_RAW_FILES = {
    "customers": "customers.csv",
    "products": "products.csv",
    "regions": "regions.csv",
    "orders": "orders.csv",
    "sales": "sales_transactions.csv",
}

print("Raw data folder:", RAW_DATA_DIR)
print("Cleaned data folder:", CLEANED_DATA_DIR)

## 3. Create Demo Data if Raw Files Are Missing

This project uses fictional retail data for portfolio demonstration.  
The synthetic data keeps the notebook runnable even when someone downloads the repository without external datasets.


In [ ]:
def raw_files_exist() -> bool:
    return all((RAW_DATA_DIR / filename).exists() for filename in EXPECTED_RAW_FILES.values())


def create_sample_raw_data(seed: int = 42) -> None:
    rng = np.random.default_rng(seed)

    customer_segments = ["Consumer", "Corporate", "Home Office", "Small Business"]
    countries = ["Germany", "France", "Netherlands", "Italy", "Spain", "Poland"]
    cities = ["Berlin", "Munich", "Paris", "Amsterdam", "Milan", "Madrid", "Warsaw", "Hamburg"]

    customers = pd.DataFrame({
        "customer_id": [f"CUST-{i:05d}" for i in range(1, 501)],
        "customer_name": [f"Customer {i}" for i in range(1, 501)],
        "customer_segment": rng.choice(customer_segments, size=500, p=[0.38, 0.27, 0.20, 0.15]),
        "email": [f"customer{i}@example.com" for i in range(1, 501)],
        "city": rng.choice(cities, size=500),
        "country": rng.choice(countries, size=500),
    })

    categories = {
        "Electronics": ["Laptops", "Phones", "Accessories"],
        "Apparel": ["Shoes", "Jackets", "Shirts"],
        "Home & Living": ["Furniture", "Kitchen", "Decor"],
        "Beauty": ["Skincare", "Haircare", "Fragrance"],
        "Sports & Outdoors": ["Fitness", "Outdoor", "Cycling"],
        "Toys & Games": ["Board Games", "Educational", "Outdoor Toys"],
    }

    product_rows = []
    product_id = 1

    for category, sub_categories in categories.items():
        for sub_category in sub_categories:
            for number in range(1, 6):
                unit_cost = round(float(rng.uniform(8, 250)), 2)
                unit_price = round(unit_cost * float(rng.uniform(1.25, 2.10)), 2)

                product_rows.append({
                    "product_id": f"PROD-{product_id:04d}",
                    "product_name": f"{sub_category} Item {number}",
                    "category": category,
                    "sub_category": sub_category,
                    "unit_cost": unit_cost,
                    "unit_price": unit_price,
                })
                product_id += 1

    products = pd.DataFrame(product_rows)

    regions = pd.DataFrame({
        "region_id": [f"REG-{i:02d}" for i in range(1, 13)],
        "region_name": [
            "North America", "North America", "Europe", "Europe",
            "Asia Pacific", "Asia Pacific", "Latin America", "Latin America",
            "Middle East & Africa", "Middle East & Africa", "Europe", "Asia Pacific"
        ],
        "country": [
            "United States", "Canada", "Germany", "France",
            "Japan", "Australia", "Brazil", "Mexico",
            "United Arab Emirates", "South Africa", "Netherlands", "Singapore"
        ],
        "state": [
            "California", "Ontario", "Bavaria", "Ile-de-France",
            "Tokyo", "New South Wales", "São Paulo", "Mexico City",
            "Dubai", "Gauteng", "North Holland", "Central"
        ],
        "city": [
            "Los Angeles", "Toronto", "Munich", "Paris",
            "Tokyo", "Sydney", "São Paulo", "Mexico City",
            "Dubai", "Johannesburg", "Amsterdam", "Singapore"
        ],
    })

    start_date = pd.Timestamp("2024-01-01")
    order_dates = start_date + pd.to_timedelta(rng.integers(0, 365, size=2000), unit="D")
    ship_dates = order_dates + pd.to_timedelta(rng.integers(1, 8, size=2000), unit="D")

    orders = pd.DataFrame({
        "order_id": [f"ORD-{i:06d}" for i in range(1, 2001)],
        "order_date": order_dates,
        "ship_date": ship_dates,
        "order_status": rng.choice(["Completed", "Returned", "Cancelled"], size=2000, p=[0.90, 0.07, 0.03]),
        "payment_method": rng.choice(["Credit Card", "PayPal", "Bank Transfer", "Invoice"], size=2000),
    })

    sales_rows = []
    sales_key = 1

    for order_id in orders["order_id"]:
        number_of_lines = int(rng.integers(1, 4))

        for _ in range(number_of_lines):
            sales_rows.append({
                "sales_id": f"SALE-{sales_key:07d}",
                "order_id": order_id,
                "customer_id": rng.choice(customers["customer_id"]),
                "product_id": rng.choice(products["product_id"]),
                "region_id": rng.choice(regions["region_id"]),
                "quantity": int(rng.integers(1, 8)),
                "discount": round(float(rng.choice([0, 0.03, 0.05, 0.10, 0.15], p=[0.45, 0.15, 0.20, 0.15, 0.05])), 2),
            })
            sales_key += 1

    sales = pd.DataFrame(sales_rows)

    customers.to_csv(RAW_DATA_DIR / EXPECTED_RAW_FILES["customers"], index=False)
    products.to_csv(RAW_DATA_DIR / EXPECTED_RAW_FILES["products"], index=False)
    regions.to_csv(RAW_DATA_DIR / EXPECTED_RAW_FILES["regions"], index=False)
    orders.to_csv(RAW_DATA_DIR / EXPECTED_RAW_FILES["orders"], index=False)
    sales.to_csv(RAW_DATA_DIR / EXPECTED_RAW_FILES["sales"], index=False)


if not raw_files_exist():
    create_sample_raw_data()
    print("Synthetic demo data created.")
else:
    print("Raw CSV files already exist.")

## 4. Load Raw Data

In [ ]:
raw_data = {
    name: pd.read_csv(RAW_DATA_DIR / filename)
    for name, filename in EXPECTED_RAW_FILES.items()
}

for name, df in raw_data.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

## 5. First Look at the Raw Data

In [ ]:
raw_data["sales"].head()

In [ ]:
raw_data["customers"].head()

## 6. Data Quality Check Before Cleaning

This step checks missing values and duplicate records before transformation.


In [ ]:
quality_summary = []

for name, df in raw_data.items():
    quality_summary.append({
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": int(df.isna().sum().sum()),
    })

quality_summary_df = pd.DataFrame(quality_summary)
quality_summary_df

## 7. Cleaning Functions

In [ ]:
def clean_text_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for column in df.select_dtypes(include="object").columns:
        df[column] = (
            df[column]
            .astype(str)
            .str.strip()
            .replace({"nan": np.nan, "None": np.nan, "": np.nan})
        )

    return df


def clean_customers(customers: pd.DataFrame) -> pd.DataFrame:
    customers = clean_text_columns(customers)
    customers = customers.drop_duplicates(subset=["customer_id"])
    customers["customer_segment"] = customers["customer_segment"].fillna("Unknown")
    customers["country"] = customers["country"].fillna("Unknown")
    return customers


def clean_products(products: pd.DataFrame) -> pd.DataFrame:
    products = clean_text_columns(products)
    products = products.drop_duplicates(subset=["product_id"])
    products["unit_cost"] = pd.to_numeric(products["unit_cost"], errors="coerce").fillna(0)
    products["unit_price"] = pd.to_numeric(products["unit_price"], errors="coerce").fillna(0)
    products["category"] = products["category"].fillna("Unknown")
    products["sub_category"] = products["sub_category"].fillna("Unknown")
    return products


def clean_regions(regions: pd.DataFrame) -> pd.DataFrame:
    regions = clean_text_columns(regions)
    regions = regions.drop_duplicates(subset=["region_id"])
    regions["region_name"] = regions["region_name"].fillna("Unknown")
    return regions


def clean_orders(orders: pd.DataFrame) -> pd.DataFrame:
    orders = clean_text_columns(orders)
    orders = orders.drop_duplicates(subset=["order_id"])
    orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
    orders["ship_date"] = pd.to_datetime(orders["ship_date"], errors="coerce")
    orders["order_status"] = orders["order_status"].fillna("Unknown")
    orders["payment_method"] = orders["payment_method"].fillna("Unknown")
    return orders.dropna(subset=["order_date"])


def clean_sales(sales: pd.DataFrame) -> pd.DataFrame:
    sales = clean_text_columns(sales)
    sales = sales.drop_duplicates(subset=["sales_id"])
    sales["quantity"] = pd.to_numeric(sales["quantity"], errors="coerce").fillna(0).astype(int)
    sales["discount"] = pd.to_numeric(sales["discount"], errors="coerce").fillna(0)
    sales = sales[sales["quantity"] > 0]
    return sales

## 8. Clean the Raw Tables

In [ ]:
customers = clean_customers(raw_data["customers"])
products = clean_products(raw_data["products"])
regions = clean_regions(raw_data["regions"])
orders = clean_orders(raw_data["orders"])
sales = clean_sales(raw_data["sales"])

cleaned_summary = pd.DataFrame([
    {"table": "customers", "rows": len(customers), "columns": customers.shape[1]},
    {"table": "products", "rows": len(products), "columns": products.shape[1]},
    {"table": "regions", "rows": len(regions), "columns": regions.shape[1]},
    {"table": "orders", "rows": len(orders), "columns": orders.shape[1]},
    {"table": "sales", "rows": len(sales), "columns": sales.shape[1]},
])

cleaned_summary

## 9. Build Dimension Tables

A star schema needs clean dimension tables and one central fact table.


In [ ]:
def add_surrogate_key(df: pd.DataFrame, key_name: str) -> pd.DataFrame:
    df = df.copy().reset_index(drop=True)
    df.insert(0, key_name, df.index + 1)
    return df


dim_customer = add_surrogate_key(customers, "customer_key")
dim_product = add_surrogate_key(products, "product_key")
dim_region = add_surrogate_key(regions, "region_key")
dim_order = add_surrogate_key(orders, "order_key")

date_range = pd.date_range(start=dim_order["order_date"].min(), end=dim_order["order_date"].max(), freq="D")

dim_date = pd.DataFrame({"full_date": date_range})
dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["day_number"] = dim_date["full_date"].dt.day
dim_date["day_name"] = dim_date["full_date"].dt.day_name()
dim_date["week_number"] = dim_date["full_date"].dt.isocalendar().week.astype(int)
dim_date["month_number"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["quarter_number"] = dim_date["full_date"].dt.quarter
dim_date["year_number"] = dim_date["full_date"].dt.year

dim_customer.head()

## 10. Build the Sales Fact Table

The fact table contains the measurable business events.  
It connects to the dimension tables through keys.


In [ ]:
fact_sales = (
    sales
    .merge(dim_order[["order_key", "order_id", "order_date"]], on="order_id", how="inner")
    .merge(dim_customer[["customer_key", "customer_id"]], on="customer_id", how="inner")
    .merge(dim_product[["product_key", "product_id", "unit_cost", "unit_price"]], on="product_id", how="inner")
    .merge(dim_region[["region_key", "region_id"]], on="region_id", how="inner")
)

fact_sales["date_key"] = fact_sales["order_date"].dt.strftime("%Y%m%d").astype(int)

fact_sales["revenue"] = fact_sales["quantity"] * fact_sales["unit_price"] * (1 - fact_sales["discount"])
fact_sales["cost"] = fact_sales["quantity"] * fact_sales["unit_cost"]
fact_sales["profit"] = fact_sales["revenue"] - fact_sales["cost"]

fact_sales["profit_margin"] = np.where(
    fact_sales["revenue"] > 0,
    (fact_sales["profit"] / fact_sales["revenue"]) * 100,
    0,
)

fact_sales = fact_sales.reset_index(drop=True)
fact_sales.insert(0, "sales_key", fact_sales.index + 1)

fact_sales = fact_sales[
    [
        "sales_key",
        "order_key",
        "customer_key",
        "product_key",
        "region_key",
        "date_key",
        "quantity",
        "unit_price",
        "discount",
        "revenue",
        "cost",
        "profit",
        "profit_margin",
    ]
]

fact_sales.head()

## 11. Final Data Quality Checks

In [ ]:
checks = {
    "fact_sales_empty": fact_sales.empty,
    "missing_revenue_values": int(fact_sales["revenue"].isna().sum()),
    "missing_profit_values": int(fact_sales["profit"].isna().sum()),
    "zero_or_negative_quantity": int((fact_sales["quantity"] <= 0).sum()),
    "missing_customer_keys": int(fact_sales["customer_key"].isna().sum()),
    "missing_product_keys": int(fact_sales["product_key"].isna().sum()),
    "missing_region_keys": int(fact_sales["region_key"].isna().sum()),
    "missing_date_keys": int(fact_sales["date_key"].isna().sum()),
}

pd.DataFrame([checks])

## 12. Executive KPI Summary

In [ ]:
total_revenue = fact_sales["revenue"].sum()
total_profit = fact_sales["profit"].sum()
profit_margin = (total_profit / total_revenue) * 100
number_of_orders = fact_sales["order_key"].nunique()
average_order_value = total_revenue / number_of_orders
customer_count = fact_sales["customer_key"].nunique()

kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Revenue",
        "Total Profit",
        "Profit Margin",
        "Number of Orders",
        "Average Order Value",
        "Customer Count",
    ],
    "Value": [
        round(total_revenue, 2),
        round(total_profit, 2),
        round(profit_margin, 2),
        number_of_orders,
        round(average_order_value, 2),
        customer_count,
    ],
})

kpi_summary

## 13. Business Analysis Tables

In [ ]:
revenue_by_region = (
    fact_sales
    .merge(dim_region[["region_key", "region_name", "country"]], on="region_key", how="left")
    .groupby(["region_name", "country"], as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        total_profit=("profit", "sum"),
    )
)

revenue_by_region["profit_margin"] = (
    revenue_by_region["total_profit"] / revenue_by_region["total_revenue"] * 100
)

revenue_by_region.sort_values("total_revenue", ascending=False).head(10)

In [ ]:
revenue_by_category = (
    fact_sales
    .merge(dim_product[["product_key", "category"]], on="product_key", how="left")
    .groupby("category", as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        total_profit=("profit", "sum"),
        quantity_sold=("quantity", "sum"),
    )
)

revenue_by_category["profit_margin"] = (
    revenue_by_category["total_profit"] / revenue_by_category["total_revenue"] * 100
)

revenue_by_category.sort_values("total_revenue", ascending=False)

In [ ]:
customer_segment_performance = (
    fact_sales
    .merge(dim_customer[["customer_key", "customer_segment"]], on="customer_key", how="left")
    .groupby("customer_segment", as_index=False)
    .agg(
        customers=("customer_key", "nunique"),
        orders=("order_key", "nunique"),
        total_revenue=("revenue", "sum"),
        total_profit=("profit", "sum"),
    )
)

customer_segment_performance["average_order_value"] = (
    customer_segment_performance["total_revenue"] / customer_segment_performance["orders"]
)

customer_segment_performance.sort_values("total_revenue", ascending=False)

## 14. Export Cleaned Tables

These exported files can be used for SQL loading, Power BI import or dashboard testing.


In [ ]:
final_tables = {
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_region": dim_region,
    "dim_date": dim_date,
    "dim_order": dim_order,
    "fact_sales": fact_sales,
}

for table_name, table in final_tables.items():
    output_path = CLEANED_DATA_DIR / f"{table_name}.csv"
    table.to_csv(output_path, index=False)

print("Cleaned tables exported successfully:")
for file in sorted(CLEANED_DATA_DIR.glob("*.csv")):
    print("-", file.name)

## 15. Business Takeaways

This cleaning process created a structured reporting layer for a Power BI dashboard.

Main results:

- Raw CSV files were cleaned and standardized.
- Duplicate records and messy text values were handled.
- Date fields were converted into a clear date dimension.
- Revenue, cost, profit and profit margin were calculated.
- A star schema structure was created with dimension tables and one fact table.
- The final tables are ready for SQL loading and Power BI reporting.

This matters because dashboards are only useful when the data behind them is clean, reliable and easy to understand.
